## Grainger Causality Matrices

In [ ]:
import os
import numpy as np
from mne.io import read_raw_eeglab

def process_set_files(input_folder):
    # Create output directory
    output_folder = "Preprocessed_CSV"
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all .set files
    set_files = [f for f in os.listdir(input_folder) if f.endswith('.set')]
    
    for set_file in set_files:
        print(f"Processing {set_file}...")
        set_path = os.path.join(input_folder, set_file)
        
        try:
            raw = read_raw_eeglab(set_path, preload=True)
        except Exception as e:
            print(f"Error loading {set_file}: {e}")
            continue
        
        # Get EEG data (exclude non-EEG channels)
        eeg_data = raw.get_data(picks='eeg')
        sfreq = raw.info['sfreq']  # Sampling frequency (should be 256Hz)
        n_channels, n_samples = eeg_data.shape
        
        # Calculate time points (0 to 1 second in steps of 1/sfreq)
        time_column = np.arange(0, 1, 1/sfreq).reshape(-1, 1)  # Column vector
        
        # Segment into 1-second chunks
        samples_per_segment = int(sfreq)
        n_segments = n_samples // samples_per_segment
        
        # Get channel names for header
        ch_names = raw.info['ch_names']
        header = "time," + ",".join(ch_names)
        
        # Base filename without extension
        base_name = os.path.splitext(set_file)[0]
        
        for seg_num in range(n_segments):
            start = seg_num * samples_per_segment
            end = start + samples_per_segment
            segment = eeg_data[:, start:end].T  # Transpose to (time, channels)
            
            # Combine time column with EEG data
            segment_with_time = np.hstack((time_column, segment))
            
            # Save as CSV
            csv_name = f"{base_name}_seg_{seg_num + 1}.csv"
            csv_path = os.path.join(output_folder, csv_name)
            
            np.savetxt(
                csv_path,
                segment_with_time,
                delimiter=',',
                header=header,
                comments='',
                fmt='%.6f'  # 6 decimal places for time and EEG values
            )
        
        print(f"Saved {n_segments} segments for {set_file}")
    
    print("All files processed successfully!")

if __name__ == "__main__":
    input_folder = r"C:\Users\umaim\Downloads\preprocessed_data"
    if not os.path.exists(input_folder):
        print(f"Error: Folder not found - {input_folder}")
    else:
        process_set_files(input_folder)

## EEG Data Preprocessing
EEG data was downloaded from an open source database (https://figshare.com/articles/dataset/EEG_Data_New/4244171) and preprocessed using EEGLAB on MATLAB. Preprocessing steps included: 
1. A band-pass filter (0.1-70 Hz) and a notch filter (50 Hz) were applied.
2. Artifact subspace reconstruction (ASR)- an automated artifact rejection method
3. Independent component analysis (ICA) is performed to remove artifacts

## Granger Causality Estimation

Use Multivariate Granger Causality (MVGC) toolbox for GC estimation. The algorithm computes MVGC using time series data in both frequency and time domains. The current code processes 19 by 19 matrices, but will be updated for 38 by 38 matrices using frequency band decomposition for final presentation

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests

def grangers_causation_matrix(data, variables, maxlag=4):
    """Generate Granger causality matrix with minimum p-values"""
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), 
                     columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r, c]], maxlag=maxlag, verbose=False)
            p_values = [round(test_result[i+1][0]['ssr_chi2test'][1], 4) for i in range(maxlag)]
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    return df

def process_eeg_files(input_folder, output_folder, maxlag=4):
    """Process all EEG files and save GC matrices with class labels"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Dictionary to store file paths by class
    class_files = {'MDD': [], 'H': []}
    
    for filepath in glob.glob(os.path.join(input_folder, '*.csv')):
        try:
            filename = os.path.basename(filepath)
            
            # Determine class from filename (adjust this based on your naming convention)
            if 'MDD' in filename or 'mdd' in filename.lower():
                class_label = 'MDD'
            else:
                class_label = 'H'
            
            # Load and process data
            data = pd.read_csv(filepath, index_col='time')
            if len(data.columns) < 2:
                continue
                
            # Generate GC matrix
            gc_matrix = grangers_causation_matrix(data, data.columns, maxlag)
            
            # Save with class label in filename
            base_name = os.path.splitext(filename)[0]
            output_path = os.path.join(output_folder, f"{class_label}_{base_name}_GCmatrix.npy")
            np.save(output_path, gc_matrix.values)
            
            class_files[class_label].append(output_path)
            
        except Exception as e:
            print(f"Error processing {filepath}: {str(e)}")
    
    # Save the file list for each class
    for class_label, files in class_files.items():
        with open(os.path.join(output_folder, f'{class_label}_files.txt'), 'w') as f:
            f.write('\n'.join(files))
    
    return class_files

# Usage
input_folder = 'Preprocessed_CSV'
output_folder = 'GC_Matrices'
class_files = process_eeg_files(input_folder, output_folder)

## Splitting data into train-test by subject number

In [ ]:
import os 
import numpy as np

folder_path = "../data/raw_data"

# List to store all loaded files
MDD_train = [] 
MDD_test = [] 

H_train = []
H_test = []

MDD_subjects = np.arange(1, 35) 
H_subjects   = np.arange(1, 31)

MDD_sub_test = np.random.choice(MDD_subjects, 7, replace=False)
MDD_sub_train = np.setdiff1d(MDD_subjects, MDD_sub_test)

H_sub_test = np.random.choice(H_subjects, 6, replace = False)
H_sub_train = np.setdiff1d(H_subjects, H_sub_test)

# Loop through every file in the folder
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    
    if filename.endswith(".npy"):
        data = np.load(file_path)  # Load the numpy array

        if filename.startswith("H"): 
            if any(f'S{sub:02d}' in filename for sub in H_sub_train): 
                H_train.append(data)

            if any(f'S{sub:02d}' in filename for sub in H_sub_test):
                H_test.append(data)
        
        if filename.startswith("MDD"):
            if any(f'S{sub:02d}' in filename for sub in MDD_sub_train):
                MDD_train.append(data)

            if any(f'S{sub:02d}' in filename for sub in MDD_sub_test):
                MDD_test.append(data)

np.save("../data/processed_data/H_train_data.npy", H_train)
np.save("../data/processed_data/H_test_data.npy", H_test)
np.save("../data/processed_data/MDD_train_data.npy", MDD_train)
np.save("../data/processed_data/MDD_test_data.npy", MDD_test)

## MLP Training + Evaluation

In [ ]:
import torch 
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, log_loss, ConfusionMatrixDisplay
import matplotlib.pyplot as plt 
import pandas as pd 

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
H_train = np.load('../data/processed_data/H_train_data.npy')
MDD_train = np.load('../data/processed_data/MDD_train_data.npy')
H_test = np.load('../data/processed_data/H_test_data.npy')
MDD_test = np.load('../data/processed_data/MDD_test_data.npy')


X_train = np.array([x.ravel() for x in np.vstack((H_train, MDD_train))])
X_test = np.array([x.ravel() for x in np.vstack((H_test, MDD_test))])
y_train = np.hstack((np.zeros(len(H_train)), np.ones(len(MDD_train))))
y_test = np.hstack((np.zeros(len(H_test)), np.ones(len(MDD_test))))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import roc_auc_score

# ChatGPT used in this section to write the flexibleMLP allowing more selection on activation functions used between each layer
class FlexibleMLP(nn.Module):
    def __init__(
        self,
        layer_sizes,
        hidden_activation="relu",
        last_hidden_activation=None,
        output_activation="sigmoid",
        dropout=0.0,
        batch_norm=False
    ):
        super().__init__()

        activation_map = {
            "relu": nn.ReLU(),
            "tanh": nn.Tanh(),
            "sigmoid": nn.Sigmoid(),
            "swish": nn.SiLU(),
            "leakyrelu": nn.LeakyReLU(),
            "none": nn.Identity()
        }

        layers = []
        num_layers = len(layer_sizes) - 1

        if isinstance(dropout, float):
            dropout = [dropout] * num_layers

        for i in range(num_layers):
            in_dim = layer_sizes[i]
            out_dim = layer_sizes[i+1]

            # Linear layer
            layers.append(nn.Linear(in_dim, out_dim))

            # Select activation
            if i < num_layers - 2:
                act = activation_map[hidden_activation.lower()]
            elif i == num_layers - 2:
                act = activation_map[last_hidden_activation.lower()] \
                      if last_hidden_activation else activation_map[hidden_activation.lower()]
            else:
                act = activation_map[output_activation.lower()]

            # Batch norm for hidden layers only
            if batch_norm and i < num_layers - 1:
                layers.append(nn.BatchNorm1d(out_dim))

            layers.append(act)

            # Dropout for hidden layers only
            if i < num_layers - 1 and dropout[i] > 0:
                layers.append(nn.Dropout(dropout[i]))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class MLPClassifier:
    def __init__(
        self,
        layer_sizes,
        hidden_activation="relu",
        last_hidden_activation=None,
        output_activation="sigmoid",
        dropout=0.0,
        batch_norm=False,
        lr=1e-3,
        batch_size=32,
        max_epochs=200,
        patience=15
    ):
        self.net = FlexibleMLP(
            layer_sizes=layer_sizes,
            hidden_activation=hidden_activation,
            last_hidden_activation=last_hidden_activation,
            output_activation="none",
            dropout=dropout,
            batch_norm=batch_norm
        )
        self.lr = lr
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience

        self.loss_fn = nn.BCEWithLogitsLoss() 
        self.loss_curve_ = []
        self.val_loss_curve_ = [] 
        self.accuracy_curve_ = []


    def fit(self, X, y,X_val=None, y_val=None):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        train_ds = torch.utils.data.TensorDataset(X, y)
        train_loader = torch.utils.data.DataLoader(
            train_ds, batch_size=self.batch_size, shuffle=True, drop_last= True
        )

        optimizer = optim.Adam(self.net.parameters(), lr=self.lr)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.2, patience=5, min_lr=1e-6
        )

        best_loss = np.inf
        patience_counter = 0


        for epoch in range(self.max_epochs):
            self.net.train()
            epoch_loss = 0.0

            for xb, yb in train_loader:
                optimizer.zero_grad()
                logits = self.net(xb).view(-1)
                loss = self.loss_fn(logits, yb)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
                self.accuracy_curve_.append(torch.sum(self.predict(xb)==yb))

            epoch_loss /= len(train_loader)
            self.loss_curve_.append(epoch_loss)

            # Validation 
            if X_val is not None:
                val_loss = self._eval_loss(X_val, y_val)
                self.val_loss_curve_.append(val_loss)
                scheduler.step(val_loss)

                if val_loss < best_loss:
                    best_loss = val_loss
                    best_state = self.net.state_dict()
                    patience_counter = 0
                else: 
                    patience_counter += 1 
                    if patience_counter >= self.patience:
                        print(f"Early stopping at epoch {epoch}")
                        break 

            else: 
                scheduler.step(epoch_loss)
            
        if X_val is not None:
            self.net.load_state_dict(best_state)

    def _eval_loss(self, X,y):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        self.net.eval()
        with torch.no_grad():
            logits = self.net(X).view(-1)
            loss = self.loss_fn(logits, y)

        return loss.item()

    def predict(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        self.net.eval()
        with torch.no_grad():
            logits = self.net(X).view(-1)
            probs = torch.sigmoid(logits)

        return (probs > 0.5).cpu().numpy().astype(int)


    def score(self, X, y):
        pred = self.predict(X)
        return np.mean(pred == y)
    
    def evaluate(self, X, y, return_auc=True):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)

        self.net.eval()
        with torch.no_grad():
            logits = self.net(X).view(-1)
            loss = self.loss_fn(logits, y)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()

        acc = (preds == y.int()).float().mean().item()

        if return_auc:
            auc = roc_auc_score(y.cpu().numpy(), probs.cpu().numpy())
            return loss.item(), acc, auc
        else:
            return loss.item(), acc


In [ ]:
model = MLPClassifier(
    layer_sizes=[361, 100, 500, 200,50, 1],
    hidden_activation='relu',
    last_hidden_activation='swish',
    output_activation='sigmoid',
    dropout=0.2,
    batch_norm=True
)
model.fit(X_train, y_train)
print('Train Accuracy: ', model.score(X_train, y_train), '\nTest Accuracy: ', model.score(X_test, y_test))
plt.plot(model.loss_curve_)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
plt.plot(model.accuracy_curve_)
plt.xlabel('Epoch')
plt.ylabel('Training Accuracy')
plt.show()

### MLP Misclassification Analysis

In [ ]:
# misclassification analysis
test_pred = model.predict(X_test)
TP = np.where((y_test == test_pred) & (test_pred ==1))[0]
TN = np.where((y_test == test_pred) & (test_pred == 0))[0]
FP = np.where((y_test != test_pred) & (test_pred == 1))[0]
FN = np.where((y_test != test_pred) & (test_pred == 0))[0]

TP_GC = X_test[np.random.choice(TP, 4)].reshape(4,19,19) + np.eye(19, dtype=float)
TN_GC = X_test[np.random.choice(TN, 4)].reshape(4,19,19) +np.eye(19, dtype=float)
FP_GC = X_test[np.random.choice(FP, 4)].reshape(4,19,19) +np.eye(19, dtype=float)
FN_GC = X_test[np.random.choice(FN, 4)].reshape(4,19,19) + np.eye(19, dtype=float)

fig, axs = plt.subplots(4, 4, figsize=(5,5))

for i, tp in enumerate(TP_GC):
    ax = axs[i][0]
    if i == 0:
        ax.set_title('TP')
    ax.imshow(tp)
    ax.set_xticks(())
    ax.set_yticks(())

for i, fp in enumerate(FP_GC): 
    ax = axs[i][1]
    if i == 0:
        ax.set_title('FP')
    ax.imshow(fp)
    ax.set_xticks(())
    ax.set_yticks(())

for i, tn in enumerate(TN_GC):
    ax = axs[i][2]
    if i == 0:
        ax.set_title('TN')
    ax.imshow(tn)
    ax.set_xticks(())
    ax.set_yticks(())
    
for i, fn in enumerate(FN_GC):
    ax = axs[i][3]
    if i == 0:
        ax.set_title('FN')
    ax.imshow(fn)
    ax.set_xticks(())
    ax.set_yticks(())

plt.suptitle('MLP Classification Samples')
plt.tight_layout()
plt.show()

### K-fold CV

In [ ]:
import os
import re
from collections import defaultdict
from sklearn.model_selection import StratifiedKFold
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

subject_files = defaultdict(list)
folder_path = "../data/raw_data"


for fname in os.listdir(folder_path):
    if not fname.endswith(".npy"):
        continue

    # Must start with H or MDD
    if not (fname.startswith("H") or fname.startswith("MDD")):
        continue

    # Extract subject number 
    match = re.search(r'S(\d{2})', fname)
    if match is None:
        continue

    subject_num = match.group(1)

    # Determine group
    group = "H" if fname.startswith("H") else "MDD"

    subject_id = f"{group}_S{subject_num}"
    subject_files[subject_id].append(fname)

subjects = np.array(list(subject_files.keys()))
labels = np.array([0 if s.startswith("H") else 1 for s in subjects])

# Parameters
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
num_epochs = 100
batch_size = 32

fold_test_accuracies = []
fold_test_aucs = []

def load_subjects(subject_ids):
    X, y = [], []

    for sid in subject_ids:
        label = 0 if sid.startswith("H") else 1
        for fname in subject_files[sid]:
            data = np.load(os.path.join(folder_path, fname))
            X.append(data)
            y.append(label)

    return np.array(X), np.array(y)


for fold, (train_sub_idx, test_sub_idx) in enumerate(skf.split(subjects, labels)):
    print(f"\n--- Fold {fold+1}/5 ---")

    train_subjects = subjects[train_sub_idx]
    test_subjects  = subjects[test_sub_idx]

    # Load data subject-wise
    X_train, y_train = load_subjects(train_subjects)
    X_test,  y_test  = load_subjects(test_subjects)

    # Optional: reshape / flatten if needed
    X_train = X_train.reshape(len(X_train), -1)
    X_test  = X_test.reshape(len(X_test), -1)


    # Build model
    model = MLPClassifier(
    layer_sizes=[X_train.shape[1], 100, 500, 200,50, 1],
    hidden_activation='relu',
    last_hidden_activation='swish',
    output_activation='sigmoid',
    dropout=0.2,
    batch_norm=False
)

    # Train
    model.fit(
        X_train, y_train, 
        X_test, y_test
    )

    # Evaluate
    results = model.evaluate(X_test,y_test)
    fold_test_accuracies.append(results[1])
    fold_test_aucs.append(results[2])

    print(f"Fold {fold+1} — Test Acc: {results[1]:.4f}, AUC: {results[2]:.4f}")



print("\n=== Subject-wise K-Fold CV Summary ===")
print(f"Accuracy: {np.mean(fold_test_accuracies):.4f} ± {np.std(fold_test_accuracies):.4f}")
print(f"AUC:      {np.mean(fold_test_aucs):.4f} ± {np.std(fold_test_aucs):.4f}")

## CNN Training + Evaluation

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs detected:", tf.config.list_physical_devices('GPU'))

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers, models

def create_cnn_model(input_shape=(19, 19, 1), learning_rate=0.001):

    model = models.Sequential([
        keras.Input(shape=input_shape),

        #First convolutional block
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        #Flatten + Dense layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),


        # Output layer
        layers.Dense(1, activation='sigmoid')
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model



def plot_training_history(history):

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Model Accuracy')
    axes[0].legend(fontsize=10)


    # Plot loss
    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Model Loss')
    axes[1].legend(fontsize=10)


    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(y_true, y_pred, class_names=None):

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def evaluate_model(model, X_test, y_test, class_names=None):

    y_pred_proba = model.predict(X_test, verbose=0)
    if len(y_pred_proba.shape) > 1 and y_pred_proba.shape[1] > 1:
        y_pred = np.argmax(y_pred_proba, axis=1)
    else:
        y_pred = (y_pred_proba > 0.5).astype(int).flatten()


    accuracy = accuracy_score(y_test, y_pred)
    print(f"Test Accuracy: {accuracy:.4f}\n")

    print(classification_report(y_test, y_pred, target_names=class_names))

    plot_confusion_matrix(y_test, y_pred, class_names)

    return y_pred



In [ ]:
# Add channel dimension
X_train = X_train[..., np.newaxis].astype(np.float32)
X_val   = X_val[..., np.newaxis].astype(np.float32)
X_test  = X_test[..., np.newaxis].astype(np.float32)  # updated to new test set

print("X_train_new:", X_train.shape)
print("y_train_new:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# Create
model = create_cnn_model(learning_rate=0.001)
model.summary()

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=40,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.4,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)

# Plot training history
plot_training_history(history)

# Evaluate on the updated test set
class_names = ['H', 'MDD']
y_pred = evaluate_model(model, X_test, y_test, class_names)

# Save model
import os
from datetime import datetime

save_dir = "/content/drive/MyDrive/phys188models"
os.makedirs(save_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f"{save_dir}/granger_cnn_model_{timestamp}.keras"
model.save(model_path)

print(f"Model saved as: {model_path}")


### CNN Hyperparameter tuning

In [ ]:
import keras_tuner as kt

def build_cnn_model(hp):
    model = keras.Sequential()

    model.add(keras.Input(shape=(19, 19, 1)))

    num_blocks = hp.Int("num_blocks", 1, 4)

    activation = hp.Choice("activation", ["relu"])

    filters_start = hp.Int("filters_start", 16, 128, step=16)

    for i in range(num_blocks):
        filters = filters_start * (2 ** i)

        model.add(layers.Conv2D(
            filters=filters,
            kernel_size=hp.Choice(f"kernel_size_{i}", [1, 5]),
            activation=activation,
            padding="same"
        ))
        model.add(layers.BatchNormalization())


        if hp.Boolean(f"pool_block_{i}"):
            model.add(layers.MaxPooling2D(pool_size=2))


        model.add(layers.Dropout(
            hp.Float(f"dropout_block_{i}", 0.0, 0.5, step=0.1)
        ))

    model.add(layers.Flatten())
    model.add(layers.Dense(
        hp.Int("dense_units", 64, 512, step=64),
        activation=activation
    ))
    model.add(layers.Dropout(
        hp.Float("dense_dropout", 0.0, 0.5, step=0.1)
    ))

    lr = hp.Float("learning_rate", 1e-5, 1e-2, sampling="log")


    optimizer = keras.optimizers.Adam(lr)


    # Output
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        loss="binary_crossentropy",
        optimizer=optimizer,
        metrics=["accuracy"]
    )

    return model


In [ ]:
tuner = kt.Hyperband(
    build_cnn_model,
    objective="val_accuracy",
    max_epochs=30,
    factor=3,
    directory="hypertuning_results",
    project_name="granger_cnn_12-12"
)


early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5
)

tuner.search(
    X_train, y_train,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    epochs=50,
    batch_size=32
)


best_hp = tuner.get_best_hyperparameters(1)[0]
model = tuner.get_best_models(1)[0]

print("Best hyperparameters:")
for hp_name in best_hp.values.keys():
    print(hp_name, ":", best_hp.get(hp_name))


In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]
model = tuner.hypermodel.build(best_hp)

early_stop_final = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=25,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_val, y_val),
    callbacks=[early_stop_final],
    verbose=1
)

plot_training_history(history)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
path =  f"{save_dir}/granger_cnn_model_{timestamp}.keras"
model.save(path)

evaluate_model(model, X_test, y_test, class_names=['H', 'MDD'])
